```markdown
# Simple Neural Network Implementation

This notebook implements a basic neural network from scratch using NumPy to solve a simple XOR-like problem. It includes functions for parameter initialization, activation functions (ReLU and Sigmoid), forward propagation, cost computation, backward propagation, parameter updates, and a training loop.
```

In [ ]:
import numpy as np
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        # randn standard normal distribution with mean 0 and standard deviation 1.
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
}
    return parameters

# The sigmoid function squashes its input to a range between 0 and 1.
# It is the only function that appears in its derivative.
# It is differentiable at every point, which helps in the effective computation of gradients during backpropagation.
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

# The Rectified Linear Unit (ReLU) function returns the input directly if it's positive, otherwise, it returns zero.
# Handling Sparse Data: ReLU helps with sparse data by zeroing out negative values, promoting sparsity and reducing overfitting.
# Faster Convergence: ReLU accelerates training by preventing saturation for positive inputs, enhancing gradient flow in deep networks.
def relu(Z):
    return np.maximum(0, Z)
def relu_derivative(Z):
    return (Z > 0).astype(int)
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

# Computes the binary cross-entropy loss.
def compute_cost(Y, A2):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(A2) + (1 - Y) * np.log(1 - A2)) / m
    # Remove single-dimensional entries from the shape of an array.
    return np.squeeze(cost)

def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]

    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m #T is transpose The transpose of a matrix is made by switching its rows with its columns, and it’s written as Aᵀ. If A is an m×n matrix, then Aᵀ is an n×m matrix.
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m

    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads
def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters
def train_neural_network(X, Y, input_size, hidden_size, output_size, epochs=1000, learning_rate=0.01):
    parameters = initialize_parameters(input_size, hidden_size, output_size)

    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, learning_rate)

        if i % 100 == 0:
            print(f"Epoch {i}: Cost = {cost}")

    return parameters
    # _ is used as a placeholder to indicate that this returned value will not be used.
def predict(X, parameters):
    A2, _ = forward_propagation(X, parameters)
    return (A2 > 0.5).astype(int)
X = np.array([[0, 2, 1, 1], [0, 1, 0, 1]])
Y = np.array([[0, 0, 0, 1]])

trained_parameters = train_neural_network(
    X, Y, input_size=2, hidden_size=4, output_size=1, epochs=10000, learning_rate=0.1)

predictions = predict(X, trained_parameters)
print("Predictions:", predictions)

This section details the individual functions used in the neural network implementation.

**1. Parameter Initialization (`initialize_parameters`)**

This function initializes the weights (W1, W2) with small random values and biases (b1, b2) with zeros. The small random initialization helps break symmetry and prevents neurons from learning the same thing. The `np.random.seed(42)` ensures reproducibility of the random initialization.

**2. Activation Functions (`sigmoid`, `relu`, `relu_derivative`)**

*   **`sigmoid(Z)`**: The sigmoid function squashes its input to a range between 0 and 1, making it suitable for the output layer of binary classification problems, as it can be interpreted as a probability.
    
    $\sigma(Z) = \frac{1}{1 + e^{-Z}}$

*   **`relu(Z)`**: The Rectified Linear Unit (ReLU) function returns the input directly if it's positive, otherwise, it returns zero. It's widely used in hidden layers due to its computational efficiency and ability to mitigate the vanishing gradient problem.
    
    $ReLU(Z) = \max(0, Z)$

*   **`relu_derivative(Z)`**: This function computes the derivative of the ReLU function, which is 1 for positive inputs and 0 for non-positive inputs. This is crucial for backpropagation.

**3. Forward Propagation (`forward_propagation`)**

This function computes the output of the neural network given an input `X` and the current `parameters`. It involves:

1.  Calculating the weighted sum of inputs plus bias for the first layer (`Z1`).
2.  Applying the ReLU activation function to `Z1` to get `A1` (activations of the hidden layer).
3.  Calculating the weighted sum of `A1` plus bias for the second layer (`Z2`).
4.  Applying the sigmoid activation function to `Z2` to get `A2` (output predictions).

The intermediate values (`Z1`, `A1`, `Z2`, `A2`) are stored in a `cache` for use during backpropagation.

**4. Cost Computation (`compute_cost`)**

This function calculates the binary cross-entropy loss, which measures how well the model's predictions (`A2`) match the true labels (`Y`). A lower cost indicates better model performance. The formula for binary cross-entropy loss is:

$L(Y, \hat{Y}) = -\frac{1}{m} \sum_{i=1}^{m} (Y_i \log(\hat{Y}_i) + (1 - Y_i) \log(1 - \hat{Y}_i))$

**5. Backward Propagation (`backward_propagation`)**

This is the core of the learning process. It computes the gradients of the cost function with respect to each parameter (weights and biases). These gradients indicate the direction and magnitude by which the parameters should be adjusted to reduce the cost. It uses the chain rule to propagate errors backward through the network.

*   `dZ2`: Derivative of cost with respect to `Z2`.
*   `dW2`, `db2`: Derivatives of cost with respect to `W2` and `b2`.
*   `dZ1`: Derivative of cost with respect to `Z1`, using the `relu_derivative`.
*   `dW1`, `db1`: Derivatives of cost with respect to `W1` and `b1`.

**6. Update Parameters (`update_parameters`)**

This function adjusts the model's parameters (weights and biases) using the calculated gradients and a `learning_rate`. The learning rate controls the step size of each update. A common update rule is:

$\text{parameter} = \text{parameter} - \text{learning_rate} \times \text{gradient}$

**7. Training Loop (`train_neural_network`)**

This function orchestrates the entire training process:

1.  Initializes parameters.
2.  Iterates for a specified number of `epochs`:
    *   Performs forward propagation to get predictions and cache.
    *   Computes the cost.
    *   Performs backward propagation to get gradients.
    *   Updates parameters using the gradients.
    *   Prints the cost every 100 epochs to monitor progress.

It returns the `trained_parameters` after all epochs.

**8. Prediction (`predict`)**

Once the neural network is trained, this function uses the learned `parameters` to make predictions on new input data `X`. It performs forward propagation and then converts the output probabilities (`A2`) into binary predictions (0 or 1) by comparing them to a threshold of 0.5.